# Session 1: Foundations & Prompt Engineering

**Great American Insurance Group — AI Developer Training**

Welcome! Over the next 90 minutes you'll go from making your first LLM call to building a prompt that extracts structured underwriting data from raw text — the same kind of pattern you'll build on in later sessions (retrieval, tools/agents, evals).

Throughout the notebook you'll see:
- 📝 **Exercise** — write or modify something yourself
- 💬 **Reflection** — a question to think about or discuss

This is not a Python course — the code here is intentionally minimal. The point is the *prompts*, not the plumbing. All LLM calls in this notebook go through one helper function, `get_client().generate(...)`, so you can focus on what you're asking the model rather than how the call is made.

Before starting, make sure you've completed the steps in this folder's `setup.md`.

## Section 1 — Setup & First Model Call

**Goal:** Confirm your environment works, make your first LLM call, and see what a response actually looks like — text, token usage, and model name.

Run the cell below. It imports the shared helper (`get_client`), asks the model to say hello, and prints back the response text plus the token usage.

In [2]:
from shared.llm import get_client

client = get_client()

result = client.generate("Say hello in one sentence.")

print("Response:", result.text)
print("Model:", result.model)
print("Tokens - input:", result.usage.input_tokens, "| output:", result.usage.output_tokens, "| total:", result.usage.total_tokens)


Response: Hello there! I hope you're having a wonderful day! 😊
Model: us.anthropic.claude-sonnet-4-6
Tokens - input: 13 | output: 18 | total: 31


📝 **Exercise 1.1** — Change the message below to something else (ask a question, request a list, whatever you like) and re-run. Notice how the token counts change based on what you asked and how long the response is.

💬 **Reflection** — `input_tokens` and `output_tokens` are billed separately and usually at different rates. Why might a provider charge more for output tokens than input tokens?

In [3]:
# Try changing this message, then re-run the cell
result = client.generate("YOUR MESSAGE HERE")

print("Response:", result.text)
print("Tokens - input:", result.usage.input_tokens, "| output:", result.usage.output_tokens)


Response: It looks like your message came through as **"YOUR MESSAGE HERE"** — which appears to be a placeholder. 😊

It seems like you may have forgotten to type your actual message!

Could you let me know:
- **What you'd like to ask or discuss?**
- **How I can help you?**

I'm ready whenever you are!
Tokens - input: 10 | output: 81


## Section 2 — Prompt Anatomy: System vs. User

**Goal:** Understand the difference between the **system prompt** (stable instructions about who the model is and how it should behave) and the **user prompt** (the task-specific input, which changes every call).

A well-structured system prompt usually includes: a **role**, a **clear task**, **relevant context**, **constraints/guardrails**, **examples** (if needed), and the **expected output format**.

Below, the same customer message is sent three ways: with no system prompt, a vague one, and a well-structured one. Run all three and compare.

In [4]:
customer_message = (
    "My basement flooded last night after the storm. There's about a foot of "
    "standing water and some of it got into the room where we keep the water heater. "
    "What do I do?"
)

# 1. No system prompt at all
no_system = client.generate(customer_message)
print("--- No system prompt ---")
print(no_system.text, "\n")

# 2. Vague system prompt
vague_system = "You are a helpful assistant."
vague = client.generate(customer_message, system_prompt=vague_system)
print("--- Vague system prompt ---")
print(vague.text, "\n")

# 3. Well-structured system prompt
structured_system = (
    "You are a claims intake assistant for a property & casualty insurer. "
    "Your job is to acknowledge the customer's situation, give brief safety guidance "
    "for water damage, and tell them the next step is to file a claim with their adjuster. "
    "Do not give a coverage determination or estimate a payout. "
    "Keep the response under 4 sentences."
)
structured = client.generate(customer_message, system_prompt=structured_system)
print("--- Structured system prompt ---")
print(structured.text)


--- No system prompt ---
This is a serious situation that needs careful handling. Here's what to prioritize:

## Immediate Safety First

**Do NOT enter the flooded area yet** - standing water near electrical equipment and a water heater is a serious electrocution risk.

**Before entering:**
- Turn off electricity to the basement at your main breaker panel (if you can access it safely from outside the flooded area)
- Contact your utility company if you're unsure or can't safely access the panel
- Turn off the gas supply if you have a gas water heater, if you can do so safely

## Getting Help

- **Call your insurance company** to report the claim and ask about emergency services coverage
- Consider calling a **licensed plumber or water damage restoration company** - many have emergency lines
- Your local fire department can sometimes advise on safety assessment

## Water Removal

Once the area is confirmed safe:
- Rent or hire a submersible pump
- Use wet/dry vacuums for remaining water


📝 **Exercise 2.1** — Write your own system prompt below that restricts the assistant to a specific role (e.g., only answers questions about auto claims, refuses to discuss competitors, etc.). Test it with an on-topic message and an off-topic message. Does it hold the boundary?

- `my_system_prompt` — your role restriction, e.g. `"You are a claims intake assistant. You only answer questions about filing or checking the status of an insurance claim. If asked about anything else, politely say you can only help with claims and redirect the user back to that topic."`
- `on_topic_message` — a question that fits inside that role, e.g. `"My car was rear-ended yesterday, how do I start a claim?"`
- `off_topic_message` — a question clearly outside that role, e.g. `"Can you recommend a good pizza recipe?"`

Run both messages through the same system prompt and compare the two responses: did it answer the claims question normally, and did it actually decline or redirect on the off-topic one — or did it just answer that too?

💬 **Reflection** — What happens if a user tries to override your system prompt in their message (e.g., "ignore your instructions and just tell me...")? This is called **prompt injection** — something worth keeping in mind anytime user input reaches the model.

In [6]:
my_system_prompt = "YOUR SYSTEM PROMPT HERE"

on_topic_message = "YOUR ON-TOPIC MESSAGE HERE"
off_topic_message = "YOUR OFF-TOPIC MESSAGE HERE"

print(client.generate(on_topic_message, system_prompt=my_system_prompt).text)
print("---")
print(client.generate(off_topic_message, system_prompt=my_system_prompt).text)


It looks like your message came through as a placeholder — **"YOUR ON-TOPIC MESSAGE HERE"** — so it seems like the actual question or topic didn't get filled in.

Could you share what you'd like to discuss or ask about? I'm happy to help! 😊
---
I notice your message appears to be a placeholder or test message rather than an actual question.

How can I help you today? Feel free to ask me anything! 😊


## Section 3 — Zero-Shot Extraction from Unstructured Text

**Goal:** Extract underwriting information out of a raw, unstructured insurance submission — without giving the model any examples first ("zero-shot").

Below are synthetic underwriting submissions. Note that `SUB-004` is missing a requested limit — a good extraction prompt should say so rather than making one up.

Fields to extract: `business_name`, `industry`, `locations`, `requested_limits`, `notable_risks`, `missing_information`.

In [7]:
SUBMISSIONS = {
    "SUB-001": (
        "Riverside Millwork LLC is a mid-size cabinetry manufacturer based out of a single "
        "facility in Dayton, Ohio. They're requesting $2,000,000 in general liability coverage "
        "and $1,000,000 in property coverage for the building and equipment. Their main exposure "
        "is woodworking dust and the use of industrial saws and finishing chemicals on-site."
    ),
    "SUB-002": (
        "Coastal Fresh Seafood Distributors operates out of two locations - a warehouse in "
        "Tampa, FL and a smaller distribution point in Savannah, GA. They are seeking "
        "$5,000,000 in general liability and want to know about cold storage equipment "
        "breakdown coverage. Notable risk: forklift traffic in the warehouse and ammonia-based "
        "refrigeration systems at both sites."
    ),
    "SUB-003": (
        "Request for BrightPath Tutoring Services, a small business with one office location "
        "in Austin, TX. They tutor K-12 students in person and want $1,000,000 in general "
        "liability and $500,000 in professional liability. No unusual physical risks noted "
        "beyond general foot traffic in the office."
    ),
    "SUB-004": (
        "Summit Ridge Landscaping, based in Denver, CO, is applying for general liability "
        "coverage. They operate commercial mowing equipment, use pesticides/herbicides, and "
        "occasionally do small retaining wall installations. They did not specify a requested "
        "coverage limit in their submission - underwriter should follow up."
    ),
    "SUB-005": (
        "Northgate Auto Body Repair operates a single shop in Columbus, OH. They're requesting "
        "$3,000,000 in general liability and $2,000,000 in garagekeepers coverage. Risks include "
        "vehicle fires during welding/paint work and hazardous waste (used oil, solvents) "
        "storage on-site."
    ),
}

for sub_id, text in SUBMISSIONS.items():
    print(f"{sub_id}: {text[:80]}...")


SUB-001: Riverside Millwork LLC is a mid-size cabinetry manufacturer based out of a singl...
SUB-002: Coastal Fresh Seafood Distributors operates out of two locations - a warehouse i...
SUB-003: Request for BrightPath Tutoring Services, a small business with one office locat...
SUB-004: Summit Ridge Landscaping, based in Denver, CO, is applying for general liability...
SUB-005: Northgate Auto Body Repair operates a single shop in Columbus, OH. They're reque...


📝 **Exercise 3.1** — Write a zero-shot extraction prompt below and run it against `SUB-001` and `SUB-004`. Look closely at what happens on `SUB-004`: does the model correctly say the requested limit is missing, or does it guess a number?

In [8]:
zero_shot_system = (
    "YOUR ZERO-SHOT EXTRACTION SYSTEM PROMPT HERE - describe the fields to extract: "
    "business_name, industry, locations, requested_limits, notable_risks, missing_information"
)

for sub_id in ["SUB-001", "SUB-004"]:
    result = client.generate(SUBMISSIONS[sub_id], system_prompt=zero_shot_system)
    print(f"--- {sub_id} ---")
    print(result.text, "\n")


--- SUB-001 ---
```json
{
  "business_name": "Riverside Millwork LLC",
  "industry": "Cabinetry Manufacturing / Millwork",
  "locations": [
    {
      "description": "Single facility",
      "city": "Dayton",
      "state": "Ohio"
    }
  ],
  "requested_limits": {
    "general_liability": "$2,000,000",
    "property": "$1,000,000",
    "notes": "Property coverage applies to building and equipment"
  },
  "notable_risks": [
    "Woodworking dust accumulation — fire and respiratory hazard",
    "Industrial saw operation — employee injury exposure",
    "Finishing chemicals on-site — flammability, storage, and environmental liability concerns",
    "Single-location concentration risk — no redundancy if facility is damaged or shut down",
    "Potential OSHA exposure related to dust control and chemical handling"
  ],
  "missing_information": [
    "Annual revenue and payroll figures for rating purposes",
    "Number of employees and any subcontractor usage",
    "Building ownership statu

## Section 4 — Few-Shot Prompting

**Goal:** See whether giving the model worked examples ("few-shot") improves consistency enough to justify the extra tokens those examples cost.

📝 **Exercise 4.1** — Below, add 1–2 worked examples (an example submission text, and the correctly extracted fields for it — including one example showing how a missing field should be handled) to your extraction prompt. Re-run against `SUB-001` and `SUB-004` and compare against Section 3's output. Also compare the `input_tokens` between the two versions — how much more did the examples cost?

In [9]:
few_shot_system = (
    zero_shot_system + "\n\n"
    "Here are two examples:\n\n"
    "Submission: BrightPath Tutoring Services, one office in Austin, TX. They tutor "
    "K-12 students in person and want $1,000,000 general liability and $500,000 "
    "professional liability. No unusual risks.\n"
    "Output: business_name: BrightPath Tutoring Services, industry: Tutoring, "
    "locations: Austin TX, requested_limits: $1,000,000 GL / $500,000 professional "
    "liability, notable_risks: none noted, missing_information: none\n\n"
    "Submission: Green Valley Petx Grooming, single location in Boise, ID. They groom "
    "dogs and cats. No coverage amount was mentioned.\n"
    "Output: business_name: Green Valley Pet Grooming, industry: Pet Grooming, "
    "locations: Boise ID, requested_limits: not specified, notable_risks: none noted, "
    "missing_information: requested coverage limit was not given"
)

for sub_id in ["SUB-001", "SUB-004"]:
    result = client.generate(SUBMISSIONS[sub_id], system_prompt=few_shot_system)
    print(f"--- {sub_id} (few-shot) ---")
    print(result.text)
    print("input_tokens:", result.usage.input_tokens, "\n")


--- SUB-001 (few-shot) ---
business_name: Riverside Millwork LLC, industry: Cabinetry Manufacturing, locations: Dayton OH, requested_limits: $2,000,000 GL / $1,000,000 property, notable_risks: woodworking dust accumulation, industrial saw operation, flammable/hazardous finishing chemicals on-site, missing_information: none
input_tokens: 334 

--- SUB-004 (few-shot) ---
business_name: Summit Ridge Landscaping, industry: Landscaping, locations: Denver CO, requested_limits: not specified, notable_risks: use of pesticides/herbicides (environmental liability exposure), operation of commercial mowing equipment (bodily injury/property damage exposure), retaining wall installations (structural/contractors liability exposure), missing_information: requested coverage limit was not provided - underwriter follow-up required
input_tokens: 312 



## Section 5 — Structured Outputs

**Goal:** So far the model has been returning free text. For downstream application code to actually use this, we need predictable JSON with an explicit schema — including how missing values should be represented (e.g., `null`).

Run the cell below to see a JSON-schema-driven prompt in action, with parsing and basic validation.

In [15]:
import json
import re

json_system = (
    "Extract underwriting information from the submission text below and return ONLY "
    "valid JSON, with no other text, matching exactly this structure:\n"
    "{\n"
    '  "business_name": string,\n'
    '  "industry": string,\n'
    '  "locations": [string],\n'
    '  "requested_limits": string or null,\n'
    '  "notable_risks": [string],\n'
    '  "missing_information": [string]\n'
    "}\n"
    "If a field is not present in the text, use null (for strings) or an empty list "
    "(for arrays), and add a note about it to missing_information. Do not invent values."
)

REQUIRED_KEYS = {"business_name", "industry", "locations", "requested_limits", "notable_risks", "missing_information"}


def strip_json_fences(text):
    """Models often wrap JSON in ```json ... ``` even when told not to - strip that defensively."""
    match = re.search(r"```(?:json)?\s*(.*?)\s*```", text, re.DOTALL)
    return match.group(1) if match else text


extracted = {}
for sub_id, text in SUBMISSIONS.items():
    result = client.generate(text, system_prompt=json_system)
    try:
        parsed = json.loads(strip_json_fences(result.text))
        missing_keys = REQUIRED_KEYS - parsed.keys()
        if missing_keys:
            print(f"{sub_id}: parsed but missing keys {missing_keys}")
        extracted[sub_id] = parsed
    except json.JSONDecodeError as e:
        print(f"{sub_id}: FAILED to parse JSON - {e}")
        print(result.text)

for sub_id, data in extracted.items():
    print(sub_id, "->", data)


SUB-001 -> {'business_name': 'Riverside Millwork LLC', 'industry': 'Cabinetry Manufacturing', 'locations': ['Dayton, Ohio'], 'requested_limits': '$2,000,000 general liability; $1,000,000 property (building and equipment)', 'notable_risks': ['Woodworking dust exposure', 'Use of industrial saws', 'Use of finishing chemicals on-site'], 'missing_information': ['Number of employees', 'Annual revenue or payroll figures', 'Year the facility was built and its square footage', 'Prior loss history', 'Details on dust collection or ventilation systems', 'Storage and handling procedures for finishing chemicals', 'Whether the business has any additional locations or operations outside the Dayton facility']}
SUB-002 -> {'business_name': 'Coastal Fresh Seafood Distributors', 'industry': 'Seafood Distribution', 'locations': ['Tampa, FL', 'Savannah, GA'], 'requested_limits': '$5,000,000 general liability', 'notable_risks': ['Forklift traffic in warehouse', 'Ammonia-based refrigeration systems at both si

📝 **Exercise 5.1** — If any submission failed to parse or was missing keys, adjust `json_system` above and re-run until every submission parses cleanly with all required keys. 💬 What did you have to add or clarify to get reliable JSON?

## Section 6 — Token Cost & Prompting Best Practices

**Goal:** Longer prompts and more examples aren't automatically better. Below is a deliberately bloated version of the JSON extraction prompt — redundant instructions, restated rules, unnecessary padding.

📝 **Exercise 6.1** — Shorten `verbose_system` below while keeping output quality on `SUB-002`. Compare `input_tokens` and output quality against the original `json_system` from Section 5.

In [9]:
verbose_system = (
    "You are an extremely helpful and knowledgeable AI assistant with expertise in "
    "the insurance underwriting domain, and you have been carefully trained to assist "
    "with a wide variety of tasks related to reading and understanding insurance submissions. "
    "When you receive an insurance submission, it is very important that you read it "
    "carefully and thoroughly before responding. Your task, which is very important, is to "
    "extract the underwriting information from the submission. Please make sure to extract "
    "the business name, and also the industry, and also the locations, and also the requested "
    "limits, and also the notable risks, and also note down any information that seems to be "
    "missing from the submission. It is critical and very important that your response is "
    "formatted as JSON, and only JSON, with absolutely no other text before or after it. "
    "The JSON should have the following keys: business_name, industry, locations, "
    "requested_limits, notable_risks, missing_information. Remember, if a field is not "
    "present in the text, use null for strings or an empty list for arrays, and please "
    "remember to add a note about it to missing_information. Please do not invent or "
    "make up any values that are not actually present in the submission text, this is "
    "very important. Thank you for your help with this important task."
)

verbose_result = client.generate(SUBMISSIONS["SUB-002"], system_prompt=verbose_system)
original_result = client.generate(SUBMISSIONS["SUB-002"], system_prompt=json_system)

print("Verbose prompt - input_tokens:", verbose_result.usage.input_tokens)
print("Original prompt - input_tokens:", original_result.usage.input_tokens)
print()
print("Verbose output:", verbose_result.text)
print()
print("Original output:", original_result.text)


Verbose prompt - input_tokens: 344
Original prompt - input_tokens: 214

Verbose output: ```json
{
  "business_name": "Coastal Fresh Seafood Distributors",
  "industry": "Seafood Distribution",
  "locations": [
    "Tampa, FL (warehouse)",
    "Savannah, GA (distribution point)"
  ],
  "requested_limits": {
    "general_liability": "$5,000,000",
    "cold_storage_equipment_breakdown": "Inquiry noted - no specific limit requested"
  },
  "notable_risks": [
    "Forklift traffic in the warehouse",
    "Ammonia-based refrigeration systems at both sites"
  ],
  "missing_information": [
    "No specific limit requested for cold storage equipment breakdown coverage",
    "Annual revenue or sales figures not provided",
    "Number of employees not provided",
    "Years in business not provided",
    "Prior loss history not provided",
    "Details on forklift operator training and safety protocols not provided",
    "Ammonia refrigeration system capacity and compliance certifications not provid

**Best practices to keep in mind going forward:**
- Define the task clearly
- Provide only relevant context
- Separate instructions from source data
- Specify the expected output format
- State how missing information should be handled
- Prevent unsupported assumptions
- Remove duplicated or conflicting instructions
- Request only the output the application actually needs
- Use few-shot examples only when they materially improve results

## Section 7 — Final Challenge

Here's a new submission you haven't seen yet. Build a prompt from scratch that:
- Uses system and user messages appropriately
- Extracts the required fields
- Returns valid JSON
- Identifies missing information (there is at least one gap in this submission)
- Avoids unsupported assumptions
- Minimizes unnecessary tokens
- Follows the best practices above

Tip: you don't have to start from a blank page — reusing and adapting your `json_system` from Section 5 is a reasonable starting point.

📝 **Exercise 7.1** — Write your final prompt below and run it against `SUB-006`.

In [10]:
SUB_006 = (
    "Lakeside Family Dentistry is requesting professional liability coverage for their "
    "single-location practice in Madison, WI. They have 4 dentists and 6 support staff. "
    "Notable risk: use of nitrous oxide sedation and on-site X-ray equipment."
)

final_system = "YOUR FINAL SYSTEM PROMPT HERE"

result = client.generate(SUB_006, system_prompt=final_system)
print(result.text)
print("input_tokens:", result.usage.input_tokens)


# Lakeside Family Dentistry – Professional Liability Coverage Assessment

---

## RISK PROFILE SUMMARY

| Factor | Detail |
|---|---|
| **Entity Type** | Single-location dental practice |
| **Location** | Madison, WI |
| **Providers** | 4 dentists (licensed) |
| **Support Staff** | 6 (hygienists, assistants, admin) |
| **Specialty** | General/Family Dentistry |
| **Notable Exposures** | Nitrous oxide sedation · Diagnostic X-ray equipment |

---

## COVERAGE ANALYSIS

### ✅ Standard Coverage Components
- **Professional Liability (Malpractice)** – per-provider and aggregate limits
- **General & Administrative Errors** – scheduling, recordkeeping, billing disputes
- **Premises Liability** – patient incidents within the clinical environment

---

### ⚠️ Elevated Risk Factors Requiring Attention

#### 1. Nitrous Oxide Sedation
> This represents the **most significant risk modifier** in this submission

- Increases exposure to **adverse sedation events**, including hypoxia or oversedation
- 

## Wrap-Up

💬 **Reflection** — Looking back at everything above, what would you monitor if this extraction prompt were running in production? Some things worth considering: how often required fields come back missing/null, how token usage grows as submissions get longer, and how consistent the output is across similar submissions.

**Next session:** we'll take this same extraction pattern and connect it to a document store — building a chat app that can answer underwriting questions over real ACORD-style submission documents using retrieval (RAG).